# Speexx AutoCycle — Colab / Kaggle

Cloud copy of the sonsole bot. Logic is identical to the local project; only the
environment layer differs (headless Chromium, no-sandbox, Linux deps).
See `CLOUD.md` in this folder.

Cells, in order:
1. **Setup** — venv, Playwright Chromium, Linux system deps
2. **Credentials** — from Kaggle Secrets / Colab userdata (`SPEEXX_ACCOUNTS`) or paste
3. **Launch** — `./start.sh <seconds>` in the background
4. **Monitor / stop** — tail the log, stop everything

Never store credentials in the notebook. Never print the accounts file.

In [ ]:
# 1) Setup — run this once per fresh Colab/Kaggle session (~2-4 min).
#    Colab: upload/clone the folder and let this cell cd into it.
#    Kaggle: if the folder arrived as a dataset it is under /kaggle/input
#    (read-only), so it is copied into /kaggle/working first.
import os, glob, shutil, subprocess

proj = os.getcwd()
if not os.path.exists("cloud_setup.sh"):
    hits = glob.glob("/kaggle/input/**/cloud_setup.sh", recursive=True)
    if hits:
        src = os.path.dirname(hits[0])
        dst = "/kaggle/working/" + os.path.basename(src)
        if not os.path.exists(dst):
            shutil.copytree(src, dst)
        proj = dst
    else:
        raise SystemExit("cloud_setup.sh not found — cd to the project folder first")
os.chdir(proj)
print("project dir:", proj)
r = subprocess.run(["bash", "cloud_setup.sh"])
assert r.returncode == 0, "setup failed — see the output above"


In [ ]:
# 2) Credentials — writes Accounts.md without ever printing it.
#    Preferred: a secret named SPEEXX_ACCOUNTS, one 'email password' per line.
import os, pathlib

accounts = None
try:                                   # Kaggle secrets
    from kaggle_secrets import UserSecretsClient
    accounts = UserSecretsClient().get_secret("SPEEXX_ACCOUNTS")
except Exception:
    pass
if not accounts:
    try:                               # Colab userdata
        from google.colab import userdata
        accounts = userdata.get("SPEEXX_ACCOUNTS")
    except Exception:
        pass
if not accounts:
    accounts = ""                       # paste 'email password' lines below instead:
    # accounts = """\
    # one@example.com its-password\n"
    # """

if accounts and accounts.strip():
    pathlib.Path("Accounts.md").write_text(accounts.strip() + "\n")
    print("Accounts.md written (%d lines)" % len(accounts.strip().splitlines()))
else:
    print("No credentials supplied — Accounts.md left empty.")

In [ ]:
# 3) Launch — same supervisor as local: staggered, per-session logs, crash cycling.
#    12600 = 3.5 h window. Colab/Kaggle kill background jobs when the notebook
#    runtime dies, so keep the window within the session's idle allowance.
import subprocess
SECONDS = 12600
subprocess.run(["bash", "start.sh", str(SECONDS)])

In [ ]:
# 4a) Monitor — the console log is the one that shows what happened in the page
!tail -n 40 ~/speexx_autocycle_multi/session1_*.log.console 2>/dev/null || echo "no session log yet"

In [ ]:
# 4b) Stop everything (browsers included) when you are done
!./stop.sh